# AgentCore Gateway Inspection

Inspect tools exposed by an MCP server configured in Bedrock AgentCore Gateway.

- **Part 1** — AWS native inspection via `bedrock-agentcore-control` API
- **Part 2** — Interactive debugging with [MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector)


## Prerequisites

- Node.js installed (for MCP Inspector)
- Python environment with `boto3`, `requests`, and `pandas`
- AWS CLI configured with appropriate credentials
- OAuth credentials for your gateway


## 1. AWS Native Inspection


### List Gateways


In [ ]:
import boto3

session = boto3.Session()  # uses default profile; set profile_name="..." if needed
client = session.client("bedrock-agentcore-control")

gateways = client.list_gateways()["items"]
for i, gw in enumerate(gateways):
    print(f"[{i}] {gw['gatewayId']} — {gw.get('name', 'N/A')}")

### Select Gateway

Pick the gateway you want to inspect from the list above.


In [ ]:
GATEWAY_INDEX = 1  # <-- change this to select a different gateway

gateway_id = gateways[GATEWAY_INDEX]["gatewayId"]
response = client.get_gateway(gatewayIdentifier=gateway_id)
print(f"Gateway: {gateway_id}")
print(f"URL: {response['gatewayUrl']}")

### Extract Connection Details


In [ ]:
import requests

gateway_url = response["gatewayUrl"]
authorizer = response["authorizerConfiguration"]["customJWTAuthorizer"]
client_id = authorizer["allowedClients"][0]
discovery_url = authorizer["discoveryUrl"]
token_url = requests.get(discovery_url).json()["token_endpoint"]

print(f"Gateway URL: {gateway_url}")
print(f"Client ID:   {client_id}")
print(f"Token URL:   {token_url}")

### List Gateway Targets

Gateway targets represent the individual tools exposed by the MCP server.


In [ ]:
import pandas as pd

targets = client.list_gateway_targets(gatewayIdentifier=gateway_id)["items"]
targets_df = pd.DataFrame(targets)
display(targets_df)

### Target Details


In [ ]:
response = client.get_gateway_target(
    gatewayIdentifier=gateway_id,
    targetId=targets_df.targetId[0],
)
response

## 2. MCP Inspector

While AWS APIs provide programmatic access to gateway metadata, 
[MCP Inspector](https://modelcontextprotocol.io/docs/tools/inspector) offers a visual interface 
for interactive debugging and rapid tool testing.


### Fetch Access Token

> ⚠️ AWS recommends storing Client Secrets in [AWS Secrets Manager](https://docs.aws.amazon.com/secretsmanager/latest/userguide/intro.html).


In [ ]:
sm = session.client("secretsmanager")
secret_id = "/FAST-stack/machine_client_secret"  # Or your Secret ID
secret = sm.get_secret_value(SecretId=secret_id)
client_secret = secret["SecretString"]
print("✅ Secret retrieved")

In [ ]:
token_response = requests.post(
    token_url,
    data=f"grant_type=client_credentials&client_id={client_id}&client_secret={client_secret}",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)
access_token = token_response.json()["access_token"]
print("✅ Token acquired. Copy it and paste into MCP Inspector's Bearer token field.")
print(f"⏳ This token expires in {token_response.json().get('expires_in', 3600)}s.")
print(access_token)

### Generate Inspector Configuration


In [ ]:
import json

inspector_config = {
    "mcpServers": {
        "unified-gateway": {
            "type": "streamable-http",
            "url": gateway_url,
        }
    }
}

config_path = "inspector-config.json"
with open(config_path, "w") as f:
    json.dump(inspector_config, f, indent=2)
print(f"Written to {config_path}")

### Launch MCP Inspector


In [ ]:
import subprocess, os, uuid

mcp_token = str(uuid.uuid4())
os.environ["MCP_PROXY_AUTH_TOKEN"] = mcp_token

process = subprocess.Popen(
    ["npx", "@modelcontextprotocol/inspector", "--config", config_path],
    env=os.environ.copy(),
)

print(f"🔗 http://localhost:6274/?MCP_PROXY_AUTH_TOKEN={mcp_token}")

### Stop Inspector


In [ ]:
import subprocess

subprocess.run(["pkill", "-f", "modelcontextprotocol/inspector"])
print("Inspector stopped")